In [ ]:
#load necessary packages
library(tidyverse)## for data processing
library(dlnm)## for lag models
library(survival)## for conditional logistic regression
library(splines) ## for non-linear splines
library(lubridate) ## processing dates
library(readr)
library(purrr) # for map function
library(data.table) # for data processing
library(mixmeta)
library(ggplot2)
options(warn=-1)
library(arrow)


In [ ]:
dir = 'case_crossover_df.feather'
df <- arrow::read_feather(dir)
df <- as.data.table(df)
for (i in 0:39) {
  column_name <- paste0("RH_lag", i)
  df[[column_name]][df[[column_name]] > 100] <- 100
}
df$age_cate <- ifelse(df$age_con<20, "0-19", ifelse(df$age_con<60, "20-59", "60+"))



In [ ]:
dir_root = "XXX"
loc_c2 = readRDS(paste0(dir_root, "location_GDP_per_capita_2010.rds"))
loc_c3 = readRDS(paste0(dir_root, "location_kgczone_IDlarge.rds"))
loc_c4 = readRDS(paste0(dir_root, "location_RDI_overall_2010_2020_IDlarge.rds"))
loc_c5 = readRDS(paste0(dir_root, "location_yearly_pop_pop.density_area.size_IDlarge.rds"))
loc_c5 = loc_c5[loc_c5$year == 2010,]
loc_c6 = read.csv('XXX/2020_walking_only_travel_time_to_healthcare_by_LocIDLarge.csv')
loc_c7 = read.csv('XXX/ac_penetration_ssp2_2010_by_LocIDLarge.csv')
colnames(loc_c2) = c("LocIDLarge", "GDP_per_capita_2010")
colnames(loc_c3) = c("lon", "lat", "ClimateZ","kgclzone", "LocIDLarge")
colnames(loc_c4) = c("LocIDLarge", "RDI_value")
colnames(loc_c5) = c("LocIDLarge", "year", "pop",  "area_size","pop_density")
colnames(loc_c6) = c("LocIDLarge", "travel_time_healthcare") # unit in minutes
colnames(loc_c7) = c("LocIDLarge", "ac_penetration") # unit: %

uni_loc = loc_c2$LocIDLarge
uni_loc = uni_loc[grepl("AUS", uni_loc) | grepl("BRA", uni_loc) | grepl("CAN", uni_loc) | grepl("CHL", uni_loc) | grepl("NZL", uni_loc)]
loc_charac = data.frame(matrix(ncol = 1, nrow = length(uni_loc)))
colnames(loc_charac) = c("LocIDLarge")
loc_charac$LocIDLarge = uni_loc
loc_charac = merge(loc_charac, loc_c2, by = "LocIDLarge", all.x = TRUE)
loc_charac = merge(loc_charac, loc_c3, by = "LocIDLarge", all.x = TRUE)
loc_charac = merge(loc_charac, loc_c4, by = "LocIDLarge", all.x = TRUE)
loc_charac = merge(loc_charac, loc_c5, by = "LocIDLarge", all.x = TRUE)
loc_charac = merge(loc_charac, loc_c6, by = "LocIDLarge", all.x = TRUE)
loc_charac = merge(loc_charac, loc_c7, by = "LocIDLarge", all.x = TRUE)
loc_charac$country = substr(loc_charac$LocIDLarge, 1, 3)
loc_charac$country = factor(loc_charac$country, levels = c("BRA", "CAN", "CHL", "NZL"))
loc_charac$country = factor(loc_charac$country, labels = c("Brazil", "Canada", "Chile", "New_Zealand"))
loc_charac = loc_charac[complete.cases(loc_charac),]

GDP_tertile = quantile(loc_charac$GDP_per_capita_2010, probs = c(1/2))
loc_charac$GDP_tertile = ifelse(loc_charac$GDP_per_capita_2010 < GDP_tertile[1], "low", "high")
RDI_tertile = quantile(loc_charac$RDI_value, probs = c(1/2))
loc_charac$RDI_tertile = ifelse(loc_charac$RDI_value < RDI_tertile[1], "low", "high")
pop_tertile = quantile(loc_charac$pop_density, probs = c(1/2))
loc_charac$pop_tertile = ifelse(loc_charac$pop_density < pop_tertile[1], "low", "high")
travel_time_tertile = quantile(loc_charac$travel_time_healthcare, probs = c(1/2))
loc_charac$travel_time_tertile = ifelse(loc_charac$travel_time_healthcare < travel_time_tertile[1], "low", "high")
ac_penetration_tertile = quantile(loc_charac$ac_penetration, probs = c(1/2))
loc_charac$ac_penetration_tertile = ifelse(loc_charac$ac_penetration < ac_penetration_tertile[1], "low", "high")

In [ ]:
heat_wave_converter <- function(df, threshold, duration, tem_lag0) {
 heat_wave_df <- data.frame(matrix(0,ncol = 44, nrow = nrow(df)))
names(heat_wave_df) <- paste0("heatwave_lag", -4:39)
#convert df > threshold to 1, the rest to 0
df_heat <- df[, (tem_lag0-4):(tem_lag0+39)]
df_heat[df_heat < threshold] <- 0
df_heat[df_heat >= threshold] <- 1
df_heat <- as.data.frame(df_heat)
for (i in 1:duration){
    heat_wave_df[,1:(44-duration)] <- heat_wave_df[,1:(44-duration)] + df_heat[i:(43-duration+i)]
}
df_heat <- heat_wave_df
names(df_heat) <- paste0("heatwave_lag", -4:39)
df_heat [df_heat != duration] <- 0
df_heat [df_heat == duration] <- 1
df_heat <- as.data.frame(df_heat)
df_heat <- df_heat[,5:44]
return(df_heat)
}

In [ ]:
get_sub_lag_rr_sum_list <- function(data_summary, threshold, duration,lag,added_effect,disease){
temperature_rm = 10
RH_rm = 7
uni_Loc = unique(data_summary$LocIDLarge)
coef_RR <- c()
vcov_RR <- c()
country_sum = c()
HDI_sum = c()
GDP_sum = c()
RDI_sum = c()
pop_sum = c
climate_sum = c()
walking_time_sum = c()
ac_penetration_sum = c()
tem_lag0 <- grep("tmean_lag0", names(data_summary))
lag_rr_sum_list <- data.frame(matrix(0,ncol = 8, nrow = 1))
names(lag_rr_sum_list) <- c("hw_or_added","disease","category","subgroup", "lag", "RR", "RR_low", "RR_high")
hoa = added_effect
sex_list = c("male","female")
age_list = c("0-19", "20-59", "60+")
country_list = c("Brazil", "Canada", "Chile", "New_Zealand")
subgroup_list = c("low", "high")
climate_list = c("A", "B", "C", "D","E")
coef_temp_subgroup = c()
vcov_temp_subgroup = c()
for (i in 1:length(uni_Loc)){
    df = data_summary[LocIDLarge == uni_Loc[i]]    
    if (nrow(df) < 100){
        next
    }else{
df_heatwave = heat_wave_converter(df, threshold, duration, tem_lag0)
df_tempearture = rowMeans(df[, tmean_temperature_lag0:(tmean_temperature_lag0+temperature_rm)])
df_heatwave = df_heatwave[[lag+1]]
df_RH = rowMeans(df[, RH_lag0:(RH_lag0+RH_rm)])
if (added_effect == "hw"){
model <- clogit(patient ~ df_heatwave + df_RH+holiday + strata(id), data = df)
}else{
model <- clogit(patient ~ df_heatwave + df_tempearture + df_RH+holiday + strata(id), data = df)
}
coef_tem_RR = summary(model)$coef[1,1]
se_tem_RR = summary(model)$coef[1,3]
}
    coef_RR = c(coef_RR, coef_tem_RR)
    vcov_RR = c(vcov_RR, se_tem_RR)
    country_sum = c(country_sum, unique(df$country))
HDI_sum = c(HDI_sum, unique(df$HDI))
GDP_sum = c(GDP_sum, unique(df$GDP))
RDI_sum = c(RDI_sum, unique(df$RDI))
pop_sum = c(pop_sum, unique(df$pop_density))
climate_sum = c(climate_sum, unique(df$kgclzone))
walking_time_sum = c(walking_time_sum, unique(df$travel_time_healthcare))
ac_penetration_sum = c(ac_penetration_sum, unique(df$ac_penetration))
}
if (length(coef_RR) > 10){   
meta_model <- mixmeta(coef_RR, vcov_RR^2 ,method = "reml")



coef_all = coef(meta_model)
vcov_all = vcov(meta_model)

coef_temp_subgroup = c(coef_temp_subgroup, coef_all)
vcov_temp_subgroup = c(vcov_temp_subgroup, vcov_all)

RR=exp(coef_all)
RR_low=exp(coef_all-1.96*sqrt(vcov_all))
RR_high=exp(coef_all+1.96*sqrt(vcov_all))
lag_rr_sum_list$hw_or_added = hoa
lag_rr_sum_list$disease = disease
lag_rr_sum_list$category = "all"
lag_rr_sum_list$subgroup = "all"
lag_rr_sum_list$lag = lag
lag_rr_sum_list$RR = as.numeric(RR)
lag_rr_sum_list$RR_low = as.numeric(RR_low)
lag_rr_sum_list$RR_high = as.numeric(RR_high)

}



#GDP
for (i in 1 : length(subgroup_list)){
index = GDP_sum == subgroup_list[i]
try({
    coef_RR_subgroup = coef_RR[index]
    vcov_RR_subgroup = vcov_RR[index]
    meta_model_subgroup <- mixmeta(coef_RR_subgroup, vcov_RR_subgroup^2 ,method = "reml")
    coef_subgroup = coef(meta_model_subgroup)
    vcov_subgroup = vcov(meta_model_subgroup)
    RR_subgroup=as.numeric(exp(coef_subgroup))
    RR_low_subgroup=as.numeric(exp(coef_subgroup-1.96*sqrt(vcov_subgroup)))
    RR_high_subgroup=as.numeric(exp(coef_subgroup+1.96*sqrt(vcov_subgroup)))
    lag_rr_sum_list <- rbind(lag_rr_sum_list, c(hoa, disease,"GDP", subgroup_list[i], lag, RR_subgroup, RR_low_subgroup, RR_high_subgroup))
    coef_temp_subgroup = c(coef_temp_subgroup, coef_subgroup)
    vcov_temp_subgroup = c(vcov_temp_subgroup, vcov_subgroup)
})
}


#RDI
for (i in 1 : length(subgroup_list)){
index = RDI_sum == subgroup_list[i]
try({
    coef_RR_subgroup = coef_RR[index]
    vcov_RR_subgroup = vcov_RR[index]
    meta_model_subgroup <- mixmeta(coef_RR_subgroup, vcov_RR_subgroup^2 ,method = "reml")
    coef_subgroup = coef(meta_model_subgroup)
    vcov_subgroup = vcov(meta_model_subgroup)
    RR_subgroup=as.numeric(exp(coef_subgroup))
    RR_low_subgroup=as.numeric(exp(coef_subgroup-1.96*sqrt(vcov_subgroup)))
    RR_high_subgroup=as.numeric(exp(coef_subgroup+1.96*sqrt(vcov_subgroup)))
    lag_rr_sum_list <- rbind(lag_rr_sum_list, c(hoa, disease,"RDI", subgroup_list[i], lag, RR_subgroup, RR_low_subgroup, RR_high_subgroup))
    coef_temp_subgroup = c(coef_temp_subgroup, coef_subgroup)
    vcov_temp_subgroup = c(vcov_temp_subgroup, vcov_subgroup)
    })
}


#pop
for (i in 1 : length(subgroup_list)){
index = pop_sum == subgroup_list[i]
try({
    coef_RR_subgroup = coef_RR[index]
    vcov_RR_subgroup = vcov_RR[index]
    meta_model_subgroup <- mixmeta(coef_RR_subgroup, vcov_RR_subgroup^2 ,method = "reml")
    coef_subgroup = coef(meta_model_subgroup)
    vcov_subgroup = vcov(meta_model_subgroup)
    RR_subgroup=as.numeric(exp(coef_subgroup))
    RR_low_subgroup=as.numeric(exp(coef_subgroup-1.96*sqrt(vcov_subgroup)))
    RR_high_subgroup=as.numeric(exp(coef_subgroup+1.96*sqrt(vcov_subgroup)))
    lag_rr_sum_list <- rbind(lag_rr_sum_list, c(hoa, disease,"pop_density", subgroup_list[i], lag, RR_subgroup, RR_low_subgroup, RR_high_subgroup))
    coef_temp_subgroup = c(coef_temp_subgroup, coef_subgroup)
    vcov_temp_subgroup = c(vcov_temp_subgroup, vcov_subgroup)
})
}


#climate
for (i in 1 : length(climate_list)){
index = climate_sum == climate_list[i]
try({
    coef_RR_subgroup = coef_RR[index]
    vcov_RR_subgroup = vcov_RR[index]
    meta_model_subgroup <- mixmeta(coef_RR_subgroup, vcov_RR_subgroup^2 ,method = "reml")
    coef_subgroup = coef(meta_model_subgroup)
    vcov_subgroup = vcov(meta_model_subgroup)
    RR_subgroup=as.numeric(exp(coef_subgroup))
    RR_low_subgroup=as.numeric(exp(coef_subgroup-1.96*sqrt(vcov_subgroup)))
    RR_high_subgroup=as.numeric(exp(coef_subgroup+1.96*sqrt(vcov_subgroup)))
    lag_rr_sum_list <- rbind(lag_rr_sum_list, c(hoa, disease,"climate", climate_list[i], lag, RR_subgroup, RR_low_subgroup, RR_high_subgroup))
    coef_temp_subgroup = c(coef_temp_subgroup, coef_subgroup)
    vcov_temp_subgroup = c(vcov_temp_subgroup, vcov_subgroup)
})
}

# walking time to healthcare
for (i in 1 : length(subgroup_list)){
index = walking_time_sum == subgroup_list[i]
try({
    coef_RR_subgroup = coef_RR[index]
    vcov_RR_subgroup = vcov_RR[index]
    meta_model_subgroup <- mixmeta(coef_RR_subgroup, vcov_RR_subgroup^2 ,method = "reml") 
    coef_subgroup = coef(meta_model_subgroup)
    vcov_subgroup = vcov(meta_model_subgroup)
    RR_subgroup=as.numeric(exp(coef_subgroup))
    RR_low_subgroup=as.numeric(exp(coef_subgroup-1.96*sqrt(vcov_subgroup)))
    RR_high_subgroup=as.numeric(exp(coef_subgroup+1.96*sqrt(vcov_subgroup)))
    lag_rr_sum_list <- rbind(lag_rr_sum_list, c(hoa, disease,"travel_time_healthcare", subgroup_list[i], lag, RR_subgroup, RR_low_subgroup, RR_high_subgroup))
    coef_temp_subgroup = c(coef_temp_subgroup, coef_subgroup)
    vcov_temp_subgroup = c(vcov_temp_subgroup, vcov_subgroup)
    })
}

# ac penetration
for (i in 1 : length(subgroup_list)){
index = ac_penetration_sum == subgroup_list[i]
try({
    coef_RR_subgroup = coef_RR[index]
    vcov_RR_subgroup = vcov_RR[index]
    meta_model_subgroup <- mixmeta(coef_RR_subgroup, vcov_RR_subgroup^2 ,method = "reml")
    coef_subgroup = coef(meta_model_subgroup)
    vcov_subgroup = vcov(meta_model_subgroup)
    RR_subgroup=as.numeric(exp(coef_subgroup))
    RR_low_subgroup=as.numeric(exp(coef_subgroup-1.96*sqrt(vcov_subgroup)))
    RR_high_subgroup=as.numeric(exp(coef_subgroup+1.96*sqrt(vcov_subgroup)))
    lag_rr_sum_list <- rbind(lag_rr_sum_list, c(hoa, disease,"ac_penetration", subgroup_list[i], lag, RR_subgroup, RR_low_subgroup, RR_high_subgroup))
    coef_temp_subgroup = c(coef_temp_subgroup, coef_subgroup)
    vcov_temp_subgroup = c(vcov_temp_subgroup, vcov_subgroup)
    })
}


#country
for (i in 1 : length(country_list)){
index = country_sum == country_list[i]
try({
    coef_RR_subgroup = coef_RR[index]
    vcov_RR_subgroup = vcov_RR[index]
    meta_model_subgroup <- mixmeta(coef_RR_subgroup, vcov_RR_subgroup^2 ,method = "reml")
    coef_subgroup = coef(meta_model_subgroup)
    vcov_subgroup = vcov(meta_model_subgroup)
    RR_subgroup=as.numeric(exp(coef_subgroup))
    RR_low_subgroup=as.numeric(exp(coef_subgroup-1.96*sqrt(vcov_subgroup)))
    RR_high_subgroup=as.numeric(exp(coef_subgroup+1.96*sqrt(vcov_subgroup)))
    lag_rr_sum_list <- rbind(lag_rr_sum_list, c(hoa, disease,"country", country_list[i], lag, RR_subgroup, RR_low_subgroup, RR_high_subgroup))
    coef_temp_subgroup = c(coef_temp_subgroup, coef_subgroup)
    vcov_temp_subgroup = c(vcov_temp_subgroup, vcov_subgroup)
    })
}

#sex

for (j in 1 : length(sex_list)){
coef_RR = c()
vcov_RR = c()
index = data_summary$sex == sex_list[j]
if (length(index) > 1){
    data_subgroup = data_summary[index]

for (i in 1:length(uni_Loc)){
    df = data_subgroup[LocIDLarge == uni_Loc[i]]    
    if (nrow(df) < 100){
        next
    }else{
df_heatwave = heat_wave_converter(df, threshold, duration, tem_lag0)
df_tempearture = rowMeans(df[, tmean_temperature_lag0:(tmean_temperature_lag0+temperature_rm)])
df_heatwave = df_heatwave[[lag+1]]
df_RH = rowMeans(df[, RH_lag0:(RH_lag0+RH_rm)])
if (added_effect == "hw"){
model <- clogit(patient ~ df_heatwave + df_RH+holiday + strata(id), data = df)
}else{
model <- clogit(patient ~ df_heatwave + df_tempearture + df_RH+holiday + strata(id), data = df)
}
coef_tem_RR = summary(model)$coef[1,1]
se_tem_RR = summary(model)$coef[1,3]
}
    coef_RR = c(coef_RR, coef_tem_RR)
    vcov_RR = c(vcov_RR, se_tem_RR)

}
try({   
meta_model <- mixmeta(coef_RR, vcov_RR^2 ,method = "reml")
coef_all = coef(meta_model)
vcov_all = vcov(meta_model)
RR = as.numeric(exp(coef_all))
RR_low = as.numeric(exp(coef_all-1.96*sqrt(vcov_all)))
RR_high = as.numeric(exp(coef_all+1.96*sqrt(vcov_all)))
lag_rr_sum_list <- rbind(lag_rr_sum_list, c(hoa, disease,"sex", sex_list[j], lag, RR, RR_low, RR_high))
    coef_temp_subgroup = c(coef_temp_subgroup, coef_all)
    vcov_temp_subgroup = c(vcov_temp_subgroup, vcov_all)
})
}
}

#age

for (j in 1 : length(age_list)){
coef_RR = c()
vcov_RR = c()
index = data_summary$age_cate == age_list[j]
if (length(index) > 1){
    data_subgroup = data_summary[index]
    for (i in 1:length(uni_Loc)){
    df = data_subgroup[LocIDLarge == uni_Loc[i]]    
    if (nrow(df) < 100){
        next
    }else{
df_heatwave = heat_wave_converter(df, threshold, duration, tem_lag0)
df_tempearture = rowMeans(df[, tmean_temperature_lag0:(tmean_temperature_lag0+temperature_rm)])
df_heatwave = df_heatwave[[lag+1]]
df_RH = rowMeans(df[, RH_lag0:(RH_lag0+RH_rm)])
if (added_effect == "hw"){
model <- clogit(patient ~ df_heatwave + df_RH+holiday + strata(id), data = df)
}else{
model <- clogit(patient ~ df_heatwave + df_tempearture + df_RH+holiday + strata(id), data = df)
}
coef_tem_RR = summary(model)$coef[1,1]
se_tem_RR = summary(model)$coef[1,3]
}
    coef_RR = c(coef_RR, coef_tem_RR)
    vcov_RR = c(vcov_RR, se_tem_RR)
}
try({   
meta_model <- mixmeta(coef_RR, vcov_RR^2 ,method = "reml")
coef_all = coef(meta_model)
vcov_all = vcov(meta_model)
RR = as.numeric(exp(coef_all))
RR_low = as.numeric(exp(coef_all-1.96*sqrt(vcov_all)))
RR_high = as.numeric(exp(coef_all+1.96*sqrt(vcov_all)))
lag_rr_sum_list <- rbind(lag_rr_sum_list, c(hoa, disease,"age", age_list[j], lag, RR, RR_low, RR_high))
    coef_temp_subgroup = c(coef_temp_subgroup, coef_all)
    vcov_temp_subgroup = c(vcov_temp_subgroup, vcov_all)
})
}
}
#get p value
lag_rr_sum_list$P_value = NA
list_category = unique(lag_rr_sum_list$category)
for (i in 1:length(list_category)){
    try({
    category = list_category[i]
    index = lag_rr_sum_list$category == category
    coef_RR_subgroup = coef_temp_subgroup[index]
    vcov_RR_subgroup = vcov_temp_subgroup[index]
    meta_model_subgroup <- mixmeta(coef_RR_subgroup, vcov_RR_subgroup ,method = "reml")
    p_value = summary(meta_model_subgroup)$qstat$pvalue
    lag_rr_sum_list$P_value[index] = p_value
    })
}

# add duration and threshold info
lag_rr_sum_list$duration = duration
lag_rr_sum_list$threshold = threshold
return(lag_rr_sum_list)
}

In [ ]:
data_summary = df

In [ ]:
disease_list <- unique(data_summary$disease)
disease_list = c("all",disease_list)
#remove the unnecessary columns
loc_charac <- loc_charac[,c("LocIDLarge","kgclzone","GDP_tertile","RDI_tertile","pop_tertile","travel_time_tertile","ac_penetration_tertile")]
#rename the columns LocIDLarg, kgclzone, HDI, GDP, RDI, pop_density
names(loc_charac) <- c("LocIDLarge","kgclzone","GDP","RDI","pop_density","travel_time_healthcare","ac_penetration")
#merge to the data_summary
data_summary <- merge(data_summary, loc_charac, by = "LocIDLarge", all.x = TRUE)

In [ ]:
# #subgroup analysis for threshold=95, duration=3, lag=0, heat wave effect
# hoa = c("hw")#, "added")
# threshold <- 97.5
# duration <- 4
# lag = 0
# # heat_wave_or_added <- "added"
# df_out <- data.frame()
# for (heat_wave_or_added in hoa){
# for (disease in disease_list){
#     if (disease == "all"){
#         data_subgroup <- data_summary
#     }else{
# subgroup_index = data_summary$disease== disease
# data_subgroup <- data_summary[subgroup_index]
#     }
# lag_rr_sum_list=lag_rr_sum_list <- data.frame(matrix(NA,ncol = 9, nrow = 1))
# names(lag_rr_sum_list) <- c("hw_or_added","disease","category","subgroup", "lag", "RR", "RR_low", "RR_high","P_value")
# try_test = try(lag_rr_sum_list <- get_sub_lag_rr_sum_list(data_subgroup, threshold, duration,lag,heat_wave_or_added,disease))
# df_out <- rbind(df_out, lag_rr_sum_list)
# write.csv(df_out, "test123.csv")
# print(disease)
# }
# }


# df_out <- df_out[complete.cases(df_out),]
# write.csv(df_out, "test123.csv")


In [ ]:
hoa = c("hw", "added")
threshold_range <- seq(90, 97.5, by=2.5)
duration_range <- c(2,3,4)

In [ ]:
df_out <- data.frame()
for (threshold in threshold_range){
for (duration in duration_range){
for (heat_wave_or_added in hoa){
for (lag in 0:8){
for (disease in disease_list){
    if (disease == "all"){
        data_subgroup <- data_summary
    }else{
subgroup_index = data_summary$disease== disease
data_subgroup <- data_summary[subgroup_index]
    }
lag_rr_sum_list=lag_rr_sum_list <- data.frame(matrix(NA,ncol = 11, nrow = 1))
names(lag_rr_sum_list) <- c("hw_or_added","disease","category","subgroup", "lag", "RR", "RR_low", "RR_high","P_value","duration","threshold")
try_test = try(lag_rr_sum_list <- get_sub_lag_rr_sum_list(data_subgroup, threshold, duration,lag,heat_wave_or_added,disease))

df_out <- rbind(df_out, lag_rr_sum_list)
write.csv(df_out, "revision_rr_sum.csv")
# print(disease)
}
}
}
}
}
df_out <- df_out[complete.cases(df_out),]
write.csv(df_out, "revision_rr_sum.csv")
